# Part 2 — MOFA tools over the Model Context Protocol (MCP)

In Part 1, the agent loop called the MOFA functions directly from Python. In this part, those same functions are provided by a separate program, an **MCP server**, and the agent accesses them by sending requests through it.

We start with our own MCP server, built around the same MOFA functions, and check that it gives the same results as Part 1. We then connect the agent to an MCP server developed by someone else, so that it can use those additional tools alongside our own within the same conversation. Finally, we look at how an MCP server like ours is actually built.

Throughout, the analysis itself stays fixed: the same MOFA functions, the same fitted model, and the same biological questions. What changes is only how the agent reaches the tools.

## Learning objectives

By the end of this notebook you should be able to:

- Explain what an MCP server is, and why moving a tool onto one changes where it runs and who can reach it, rather than what it does.
- Connect an agent to an MCP server and use its tools exactly as you used local tools in Part 1.
- Connect the same agent to a server written and run by someone else, and use both sets of tools in one conversation.
- Describe what information leaves your machine when an external server is involved.

## What is MCP?

An **MCP server** is a program that makes a set of tools available to other programs. It runs on its own, holds whatever data those tools need, and waits for requests. An agent that wants to use one of those tools does not import any code: it connects to the server, asks what tools are available, and asks for one to be run.

For that exchange to work, the agent and the server have to agree on how to ask those questions. The **Model Context Protocol (MCP)** is that agreement. It defines how a client asks a server what it offers, how it requests that a tool be run, and how results come back. Because the protocol is the same everywhere, any MCP-compatible agent can use any MCP server.

You may ask: why add this extra layer at all? It is useful because it separates the tool provider from the application using the tools. The same set of tools can then be reused by different notebooks or agents without each one having to integrate the underlying code itself, and access to the data and functions can stay behind a single controlled interface. Just as importantly, the same standard lets your agent connect to tools built by other people without needing a new custom integration for each one.

The server we use here is `server/mofa_mcp_server.py`. We wrote it for this practical: it holds the same eight analysis functions from Part 1, backed by the same cached MOFA model. This notebook is about using it. The last section shows how it was built.

## 0. Setup

Load the API key and locate the MCP server script.

In [ ]:
from pathlib import Path
import os, sys, json

from dotenv import load_dotenv

if Path.cwd().name == "notebooks":
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")

if not os.environ.get("ANTHROPIC_API_KEY"):
    raise RuntimeError("ANTHROPIC_API_KEY not found. Add it to a .env in the project root.")

SERVER_PATH = PROJECT_ROOT / "server" / "mofa_mcp_server.py"
assert SERVER_PATH.exists(), f"MCP server not found at {SERVER_PATH}"

print("API key loaded:", bool(os.environ.get("ANTHROPIC_API_KEY")))
print("MCP server    :", SERVER_PATH.name)

API key loaded: True
MCP server    : mofa_mcp_server.py


## 1. Connecting to the server

This section sets up the connection between the notebook and the MCP server, then rebuilds the same agent loop used in Part 1.

`MultiServerMCPClient` is an MCP client: it launches the server, performs the exchange described above, and hands back the server's tools in the form LangChain expects.

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient({
    "mofa": {
        "command": sys.executable,
        "args": [str(SERVER_PATH)],
        "transport": "stdio",       # the server runs as a subprocess of this notebook
    }
})

# Ask the server what it offers. Each tool arrives with its name, its arguments
# and its description, the same three things Claude saw in Part 1.
tools = await client.get_tools()
tools_by_name = {t.name: t for t in tools}

print(f"{len(tools)} tools from the MOFA server:")
for t in tools:
    print(f"  {t.name}")

8 tools from the MOFA server:
  data_summary
  split_summary
  active_factors
  factor_view_r2
  factor_subtype_association
  top_features_for_factor
  classify_subtype_from_factors
  train_vs_test_subtype_association


Binding and the agent loop are unchanged from Part 1. The only difference is that the tools now live in another program, so calling one means sending a request and waiting for the reply — which is what `await` and `ainvoke` mark on the tool call.

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage

MODEL = "claude-haiku-4-5"
llm = ChatAnthropic(model=MODEL, temperature=0)
llm_with_tools = llm.bind_tools(tools)

# Identical to the system prompt in Part 1.
SYSTEM = ("You are a computational-biology assistant analysing a fitted MOFA model "
          "of TCGA breast-cancer multi-omics data. Use the tools to gather evidence; "
          "ground every quantitative claim in tool results and name which tool you used. "
          "Factors are named 'Factor1'..'Factor10'; subtypes are PAM50 (LumA, LumB, "
          "Basal, Her2, Normal).")


async def run_agent(question: str, max_steps: int = 6, verbose: bool = True) -> AIMessage:
    """
    Identical to Part 1's run_agent, except that it waits for the server to reply.
    """
    messages = [SystemMessage(content=SYSTEM), HumanMessage(content=question)]
    for step in range(max_steps):

        # Ask the model what to do next.
        model_response = await llm_with_tools.ainvoke(messages)
        messages.append(model_response)

        # If no tool was requested, the model has finished its answer. Return it.
        if not model_response.tool_calls:
            return model_response

        # Otherwise, run each requested tool
        for call in model_response.tool_calls:
            if verbose:
                print(f"[step {step}] -> {call['name']}({call['args']})")
            # ...except that the tool does not run here: the request goes to the 
            # server, which runs the function and sends the result back.
            result = await tools_by_name[call["name"]].ainvoke(call["args"])
            messages.append(ToolMessage(content=json.dumps(result, default=str),
                                        tool_call_id=call["id"]))
    raise RuntimeError(f"Agent did not finish within {max_steps} steps.")

print("Bound", len(tools), "MCP tools to", MODEL)

Bound 8 MCP tools to claude-haiku-4-5


<div style="border-left:4px solid #028090; background:#f0f7f8; color:#16242f;
            padding:0.75em 1em; margin:1em 0; border-radius:4px;">
<details>
<summary style="cursor:pointer; font-weight:bold;">Note — why does <code>run_agent()</code> need <code>async</code>, <code>await</code> and <code>ainvoke</code> now?</summary>

<p style="margin-top:0.9em;">In Part 1, tools were called like any other Python function:</p>

<pre style="background:#ffffff; padding:0.6em 0.8em; border-radius:3px; margin:0.6em 0;"><code>result = tools_by_name[call["name"]].invoke(call["args"])</code></pre>

<p>In Part 2, the same line reads:</p>

<pre style="background:#ffffff; padding:0.6em 0.8em; border-radius:3px; margin:0.6em 0;"><code>result = await tools_by_name[call["name"]].ainvoke(call["args"])</code></pre>

<p>This is because the agent is no longer calling a function in this notebook. An MCP tool may be running in another program, so the notebook sends it a request and waits for the result to come back.</p>

<p>Python has a way of handling that kind of waiting without stopping everything else: <em>asynchronous</em> programming (<code>async</code>). An asynchronous function can pause while it waits for a result and let Python get on with other work in the meantime — useful whenever a program is talking to several things at once.</p>

<p>That is what the two new keywords are. The <code>a</code> in <code>ainvoke</code> means asynchronous: it starts the operation rather than completing it. <code>await</code> then means, roughly, <em>pause here until the result is ready, then continue</em>.</p>

<p>Python only allows <code>await</code> inside an asynchronous function, so <code>run_agent()</code> has to change too — from <code>def run_agent(...)</code> to <code>async def run_agent(...)</code>.</p>

<p style="margin-bottom:0;">We gain nothing from this here: the notebook asks one question at a time and has no other work to get on with while it waits. The library that connects us to MCP servers is simply written this way, because it is built for programs handling several servers and many requests at once.</p>

</details>
</div>

## 2. The same questions, answered through the server

Below are two of the questions from Part 1: the first one, which needs a single tool, and the flagship question, which needs several. The tool names in the trace and the answers themselves should match what you saw in Part 1.

In [ ]:
# Q1 from Part 1: one tool
query = "How many patients and features are in each omics view, and how many patients per PAM50 subtype?"
answer = await run_agent(query, max_steps=12)
print(answer.content)

[step 0] -> data_summary({})
Based on the **data_summary** tool results:

**Patients and Features per Omics View:**
- **Transcriptomics**: 603 patients, 29,995 features
- **Proteomics**: 603 patients, 464 features
- **Methylation**: 603 patients, 200,000 features

**Patients per PAM50 Subtype:**
- **LumA** (Luminal A): 322 patients
- **LumB** (Luminal B): 118 patients
- **Basal**: 97 patients
- **Her2**: 41 patients
- **Normal**: 25 patients

The dataset contains 603 patients total across all three omics views, with Luminal A being the most common subtype and Normal being the least common.


In [ ]:
# Q5 from Part 1, the flagship: several tools, the second chosen after seeing the first result
query = "Which MOFA factor is most associated with breast-cancer subtype, and which transcriptomic features most strongly drive it?"
answer = await run_agent(query, max_steps=12)
print(answer.content)

[step 0] -> factor_subtype_association({})
[step 1] -> top_features_for_factor({'factor': 'Factor2', 'view': 'transcriptomics', 'n': 5})
## Summary

**Factor2** is the MOFA factor most strongly associated with breast-cancer PAM50 subtype, with an eta-squared association of **0.743** (from `factor_subtype_association`).

The **top 5 transcriptomic features** most strongly driving Factor2 are:

**Positive drivers** (highest weights):
1. **ENSG00000160182.3** (weight: 0.47)
2. **ENSG00000173467.9** (weight: 0.469)
3. **ENSG00000082175.15** (weight: 0.436)
4. **ENSG00000235687.9** (weight: 0.424)
5. **ENSG00000160180.15** (weight: 0.417)

**Negative drivers** (lowest weights):
1. **ENSG00000166535.20** (weight: -0.337)
2. **ENSG00000186832.9** (weight: -0.301)
3. **ENSG00000102243.13** (weight: -0.283)
4. **ENSG00000185686.18** (weight: -0.279)
5. **ENSG00000135069.14** (weight: -0.278)

These genes represent the key transcriptomic signatures that Factor2 captures to distinguish between br

Same tools, same order, same answer. The functions ran in another process, and neither Claude nor the agent loop had to be told anything about that.

Now write your own question. Anything the eight tools can answer between them — a good one needs more than a single tool, so that you can watch Claude decide what to ask for next.

In [ ]:
#################################################
# Write a question that needs more than one tool. 
# Additional clue: look at Part 1 notebook for ideas
query = "Is the factor that explains the most variance also the one that best separates subtypes? " \
        "Compare the R2 ranking with the subtype association ranking and explain any mismatch."

answer = await run_agent(query, max_steps=12)
print(answer.content)
#################################################

[step 0] -> active_factors({})
[step 0] -> factor_subtype_association({})
Excellent! Now I can provide a detailed comparison. Here's what the data shows:

## R2 Ranking (Variance Explained - Total Across All Views)
1. **Factor1**: 38.11%
2. **Factor2**: 23.11%
3. **Factor3**: 16.92%
4. **Factor4**: 11.41%
5. **Factor5**: 8.56%
6. **Factor6**: 6.78%
7. **Factor7**: 6.73%
8. **Factor8**: 4.87%
9. **Factor9**: 3.82%
10. **Factor10**: 3.01%

## Subtype Association Ranking (η² - Association with PAM50 Subtype)
1. **Factor2**: η² = 0.743
2. **Factor1**: η² = 0.350
3. **Factor7**: η² = 0.264
4. **Factor8**: η² = 0.173
5. **Factor4**: η² = 0.146
6. **Factor5**: η² = 0.080
7. **Factor3**: η² = 0.075
8. **Factor10**: η² = 0.046
9. **Factor6**: η² = 0.036
10. **Factor9**: η² = 0.003

## Key Findings: **No, they are NOT the same!**

**The mismatch is striking:**
- **Factor1** dominates in variance explained (38.11%, rank #1) but ranks only **#2** in subtype association (η² = 0.350)
- **Factor2** i

## 3. Connecting to somebody else's server

Everything so far used a server we wrote, fronting our own model. The more useful case is a server somebody else runs, exposing data we do not have.

[BioMCP](https://biomcp.org) is one: an open-source MCP server covering around fifteen public biomedical sources — PubMed, ClinicalTrials.gov, ClinVar, MyGene.info and others — behind one set of tools. Because it speaks the same protocol, connecting to it takes the same few lines as connecting to our own server, and the agent can use both at once.

It is already installed in this environment. Elsewhere, you would install it with:

```bash
pip install biomcp-python
```

The next cell finds the BioMCP command and adds it as a second entry alongside our own server.

In [ ]:
import shutil, subprocess

BIOMCP_BIN = shutil.which("biomcp")
assert BIOMCP_BIN, "biomcp not found on PATH -- pip install biomcp-python, then restart the kernel"

# The CLI has used both `serve` and `run` across versions; pick whichever this one has.
help_text = subprocess.run([BIOMCP_BIN, "--help"], capture_output=True, text=True).stdout
subcommand = "run" if " run " in help_text else "serve"

client = MultiServerMCPClient({
    "mofa":   {"command": sys.executable, "args": [str(SERVER_PATH)], "transport": "stdio"},
    "biomcp": {"command": BIOMCP_BIN,     "args": [subcommand],       "transport": "stdio"},
})

mofa_tool_names = set(tools_by_name)
all_tools = await client.get_tools()
all_tools_by_name = {t.name: t for t in all_tools}

print(f"{len(all_tools)} tools in total")
print("  ours   :", sorted(mofa_tool_names))
print("  theirs :", sorted(set(all_tools_by_name) - mofa_tool_names))

11 tools in total
  ours   : ['active_factors', 'classify_subtype_from_factors', 'data_summary', 'factor_subtype_association', 'factor_view_r2', 'split_summary', 'top_features_for_factor', 'train_vs_test_subtype_association']
  theirs : ['biomcp', 'get', 'search']


With two sets of tools available, the system prompt has to say which is for what. It also has to mention one practical detail: our MOFA tools return Ensembl IDs with a version suffix, and BioMCP does not recognise those.

In [ ]:
llm_with_all_tools = llm.bind_tools(all_tools)

SYSTEM_MULTI = (
    "You are a computational-biology assistant. You have two kinds of tools: "
    "(1) MOFA tools analysing a fitted multi-omics model of TCGA breast-cancer "
    "data (factors 'Factor1'..'Factor10', PAM50 subtypes), and (2) BioMCP tools "
    "for general biomedical knowledge (genes, drugs, diseases, literature). "
    "Use MOFA tools for questions about our fitted model/factors/patients; use "
    "BioMCP tools for general biomedical facts about specific genes, drugs, or "
    "diseases. Ground every claim in tool output and name which tool you used. "
    "NOTE: MOFA tools return Ensembl gene IDs with a version suffix (e.g. "
    "'ENSG00000160180.15'). BioMCP tools do not recognise the version suffix -- "
    "strip it (to 'ENSG00000160180') before passing an ID to any BioMCP tool. "
    "Resolve one gene at a time rather than issuing many lookups in parallel. "
    "If a question asks for something no available tool result actually supports, "
    "say so explicitly rather than inferring an answer from general biomedical knowledge."
)


async def run_agent_multi(question: str, max_steps: int = 12, verbose: bool = True) -> AIMessage:
    """Same loop as before, over the combined tool set."""
    messages = [SystemMessage(content=SYSTEM_MULTI), HumanMessage(content=question)]
    for step in range(max_steps):

        model_response = await llm_with_all_tools.ainvoke(messages)
        messages.append(model_response)

        if not model_response.tool_calls:
            return model_response

        for call in model_response.tool_calls:
            if verbose:
                print(f"[step {step}] -> {call['name']}({call['args']})")
            result = await all_tools_by_name[call["name"]].ainvoke(call["args"])
            if verbose:
                # Worth watching: the arguments above are what leaves your machine.
                preview = json.dumps(result, default=str)
                print(f"           <- {preview[:200]}"
                      + (f"... [{len(preview)} chars]" if len(preview) > 200 else ""))
            messages.append(ToolMessage(content=json.dumps(result, default=str),
                                        tool_call_id=call["id"]))
    raise RuntimeError(f"Agent did not finish within {max_steps} steps.")

print("Bound", len(all_tools), "tools to", MODEL)

Bound 11 tools to claude-haiku-4-5


The question below cannot be answered by either server alone. Finding the genes that drive a factor needs our model; saying what those genes do needs the literature.

In [ ]:
query = "Factor2 is associated with breast cancer subtype. What biological processes " \
        "are enriched among its top 50 genes?"
answer = await run_agent_multi(query, max_steps=12)
print(answer.content)

[step 0] -> top_features_for_factor({'factor': 'Factor2', 'n': 50, 'view': 'transcriptomics'})
           <- [{"type": "text", "text": "{\n  \"factor\": \"Factor2\",\n  \"view\": \"transcriptomics\",\n  \"top_negative\": {\n    \"ENSG00000166535.20\": -0.337,\n    \"ENSG00000186832.9\": -0.301,\n    \"ENSG00... [3800 chars]
[step 1] -> biomcp({'command': 'enrich ENSG00000166535,ENSG00000186832,ENSG00000102243,ENSG00000185686,ENSG00000135069,ENSG00000102854,ENSG00000105141,ENSG00000164434,ENSG00000143546,ENSG00000136928,ENSG00000205420,ENSG00000156219,ENSG00000107159,ENSG00000176887,ENSG00000198729,ENSG00000280916,ENSG00000163362,ENSG00000163220,ENSG00000173894,ENSG00000026559,ENSG00000189001,ENSG00000196611,ENSG00000229544,ENSG00000143452,ENSG00000163064,ENSG00000117148,ENSG00000135374,ENSG00000167656,ENSG00000105173,ENSG00000114805,ENSG00000101057,ENSG00000188910,ENSG00000144354,ENSG00000166426,ENSG00000119547,ENSG00000019169,ENSG00000130294,ENSG00000258910,ENSG00000069482,ENSG0000014

Watch the trace: it starts in our tools, then crosses into theirs, using the gene IDs the first set returned.

Now write your own. Something that starts in the MOFA model and ends in the public data.

In [ ]:
#################################################
# Write a question that has to use both sets of tools. 
# Additional clue: get a gene or a set of genes out of the MOFA model first, then ask something 
# about them that only public data can answer -- clinical trials, variants, or published literature.
query = "Run BioMCP's gene-set enrichment on Factor2's top 5 genes and report only the " \
        "pathways/processes it returns."
answer = await run_agent_multi(query, max_steps=12)
print(answer.content)
#################################################

[step 0] -> top_features_for_factor({'factor': 'Factor2', 'n': 5, 'view': 'transcriptomics'})
           <- [{"type": "text", "text": "{\n  \"factor\": \"Factor2\",\n  \"view\": \"transcriptomics\",\n  \"top_negative\": {\n    \"ENSG00000166535.20\": -0.337,\n    \"ENSG00000186832.9\": -0.301,\n    \"ENSG00... [560 chars]
[step 1] -> biomcp({'command': 'enrich ENSG00000160180,ENSG00000235687,ENSG00000082175,ENSG00000173467,ENSG00000160182'})
           <- [{"type": "text", "text": "# Enrichment: ENSG00000160180, ENSG00000235687, ENSG00000082175, ENSG00000173467, ENSG00000160182\n\n| Source | ID | Name | p-value |\n|--------|----|------|---------|\n| WP... [963 chars]
## Gene-Set Enrichment Results for Factor2's Top 5 Genes

**Pathways and Processes:**

1. **Pancreatic cancer subtypes** (WikiPathways, p=5.456e-5)
2. **Estrogen-dependent gene expression** (Reactome, p=4.234e-4)
3. **ESR-mediated signaling** (Reactome, p=1.333e-3)
4. **Maintenance of gastrointestinal epithelium** (GO Biol

## 4. What this bought us

The first half of the notebook changed nothing about what the agent could do. That was the point: the same tools, reached differently, gave the same answers. What changed is that the tools and the data behind them now live in one place, loaded once, and any number of clients can use them without a copy of the code or the omics tables.

The second half is where we see the usefulness of MCPs. Connecting to BioMCP took the same handful of lines as connecting to our own server, and the agent gained tools over fifteen public databases that we did not write and do not maintain.

**What leaves your machine.** The omics tables and the fitted model stay inside the MOFA server's process; BioMCP never sees them. What reaches BioMCP is whatever Claude puts in a tool call's arguments — in the traces above, gene identifiers — which BioMCP then passes on to the public APIs behind it. Separately, and true of Part 1 as well, the whole conversation goes to Anthropic as part of the agent loop.

For this dataset that is low-stakes: TCGA is public and de-identified. On data that is not, note that the system prompt is a guideline and not a boundary — **nothing prevents Claude from putting more context into a tool call than you intended**. If you need a guarantee, validate the arguments in code before they reach the server.

## Reflection

- Compare the traces in Section 2 with the ones from Part 1. Did Claude call the same tools, in the same order?
- The agent loop never mentions MCP. Why did it not need changing when the tools moved onto a server?
- When you asked a question needing both servers, how did Claude get from a MOFA result to a BioMCP query?
- Our server exposes only read-only tools. What would you want to change if one of them wrote to a file?

## 5. How the server works

*For the interested reader. Nothing here is needed to run anything above.*

### Building the server

MCP has official software development kits in several languages. Ours is written with the Python one, using its high-level interface, **FastMCP**, which turns ordinary functions into MCP tools. The whole idea fits in a few lines:

```python
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("my-server")

@mcp.tool()
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

if __name__ == "__main__":
    mcp.run()
```

That is all MCP asks of you. Everything else in `server/mofa_mcp_server.py` is the MOFA analysis from Part 1, unchanged.

The decorator is what does the work. `@mcp.tool()` takes the function's name, its type-hinted arguments, and its docstring, and makes those three things visible to any client that connects. They are the same three things Claude saw in Part 1, and the same job LangChain's `@tool` did there — one layer further out. As in Part 1, the docstring is the interface: it is what the model reads when deciding whether the tool is worth calling, and the body is what it never sees.

Everything above the decorators in our file — loading the omics tables, the train/test split, opening the fitted model, projecting the test patients — runs once, when the server starts. That is what makes the reuse described earlier practical: one copy of the data, loaded once, shared by every client that connects, rather than a copy loaded inside every notebook.

Two other decorators publish things a plain function list cannot. `@mcp.resource("mofa://summary")` publishes data addressed by a URI — here a compact description of the fitted model — which the application reads into context rather than the model choosing to call. `@mcp.prompt()` publishes a reusable template that the user picks; ours is `interpret_factor(factor)`, four lines that ask for a full interpretation of one factor in a fixed form. A prompt supplies instructions rather than capability, which is an idea Part 3 takes much further.

Each tool also carries `ToolAnnotations(readOnlyHint=True, ...)`, saying it only reads and never changes anything. A client can use that to run a tool without pausing to ask a human first — worth having precisely because the caller here is a model rather than a fixed script.

Finally, `mcp.run()` at the bottom is what lets `MultiServerMCPClient` start the server with `python server/mofa_mcp_server.py`. Run that command yourself in a terminal and it will sit there apparently doing nothing, waiting for input — which is what a server of this kind looks like when nothing is talking to it.

### The exchange underneath

Now that you have seen how a server publishes things, here is what the client and the server actually say to each other.

Below we connect with MCP's own client library instead of the adapter. The pattern is always the same: open a session, `initialize` it — the server replies with what it can do — and then ask for things. No model is involved, so this costs nothing to run.

In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server_params = StdioServerParameters(command=sys.executable, args=[str(SERVER_PATH)])

async with stdio_client(server_params) as (read, write):
    async with ClientSession(read, write) as session:
        init = await session.initialize()
        print("connected to:", init.serverInfo.name, "\n")

        # TOOLS -- the model may choose to call these
        for t in (await session.list_tools()).tools:
            args = ", ".join(t.inputSchema.get("properties", {}))
            print(f"  tool      {t.name}({args})")

        # RESOURCES -- data to read into context, not called by the model
        for r in (await session.list_resources()).resources:
            print(f"  resource  {r.uri}")

        # PROMPTS -- reusable templates a user can pick
        for p in (await session.list_prompts()).prompts:
            args = ", ".join(a.name for a in (p.arguments or []))
            print(f"  prompt    {p.name}({args})")

        print("\nmofa://summary ->")
        summary = await session.read_resource("mofa://summary")
        print(" ", summary.contents[0].text)

        print("\ncalling factor_subtype_association ->")
        called = await session.call_tool("factor_subtype_association", {})
        print(" ", called.content[0].text[:150], "...")

connected to: eccb2026-mofa 

  tool      data_summary()
  tool      split_summary()
  tool      active_factors()
  tool      factor_view_r2(factor)
  tool      factor_subtype_association()
  tool      top_features_for_factor(factor, view, n)
  tool      classify_subtype_from_factors()
  tool      train_vs_test_subtype_association()
  resource  mofa://summary
  prompt    interpret_factor(factor)

mofa://summary ->
  {
  "n_patients": 603,
  "views": [
    "transcriptomics",
    "proteomics",
    "methylation"
  ],
  "n_factors": 10,
  "active_factors": [
    "Factor1",
    "Factor2",
    "Factor3",
    "Factor4",
    "Factor5",
    "Factor6",
    "Factor7",
    "Factor8",
    "Factor9",
    "Factor10"
  ],
  "subtypes": {
    "LumA": 322,
    "LumB": 118,
    "Basal": 97,
    "Her2": 41,
    "Normal": 25
  }
}

calling factor_subtype_association ->
  {
  "factor": "Factor2",
  "eta_squared": 0.743
} ...


Two things that makes visible.

**A server offers more than tools.** *Tools* are chosen by the model. *Resources* are data the application reads into context, like `mofa://summary`. *Prompts* are templates the user picks, like `interpret_factor`. In Section 1 we called `get_tools()`, which pulled in only the first of the three, because that is what a tool-calling loop needs; `get_resources()` and `get_prompt()` fetch the others.

**The messages are ordinary.** They travel as JSON-RPC — a plain request/response format — over a transport. Ours is `stdio`: the server runs as a subprocess and the two talk through its standard input and output, the same way you would pipe one command-line program into another. Over a network it would be HTTP instead, and nothing above would change.